# HV-007 — Independent Blind Validation

**Technical gate:** IV-001  
**Dataset release:** VSM-LAB-DATA-v1.1

This notebook performs the first independent evaluation of the frozen calibration model on reserved STATIC / VALIDATION measurements.

The primary rule is non-negotiable:

> The C1 model identity and coefficient are frozen. VALIDATION results may be observed, but they may not be used to refit, retune, or replace C1.

## 1. Engineering Question

> Does the frozen C1 calibration model improve agreement between the DUT and reference on the reserved STATIC / VALIDATION data, without refitting or retuning?

This notebook evaluates **independent static generalization** only.

Final HVAC suitability remains deferred until dynamic response, stability/drift, and measurement uncertainty are evaluated.

## 2. Blind-Validation Governance

Before this notebook is run:

- C1 is already frozen from HV-006;
- C1 offset is fixed at approximately +1.2266788194 %RH;
- C5 is frozen separately as a Direction/state diagnostic;
- no VALIDATION measurement has been used to select or tune these models;
- hidden simulator truth remains prohibited.

The validation results below are evidence, not a new training set.

## 3. Reproducible Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.validation_analysis import (
    load_frozen_calibration_records,
    load_static_validation,
    summarize_validation_test_ids,
)

tables_dir = project_root / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

Project root: /home/runner/work/hvac-sensor-validation-metrology/hvac-sensor-validation-metrology


## 4. IV-01 — VALIDATION Boundary Audit

Expected design from the public test matrix:

- 36 VALIDATION Test_IDs;
- 60 samples per Test_ID;
- 2,160 measurement rows;
- RH = 30, 50, 70 %RH;
- Temperature = 15, 25, 35 °C;
- Direction = UP, DOWN;
- Day = 3;
- Run = 1, 2.

This section is the first point at which reserved VALIDATION measurement rows are loaded.

In [2]:
validation_df, test_matrix_df = load_static_validation(project_root)

validation_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "VALIDATION")
].copy()

print("VALIDATION boundary audit")
print("-------------------------")
print(f"Measurement rows      : {len(validation_df)}")
print(f"Measurement Test_IDs  : {validation_df['Test_ID'].nunique()}")
print(f"Matrix Test_IDs       : {validation_matrix_df['Test_ID'].nunique()}")
print(
    "Samples/Test_ID      : "
    f"{sorted(validation_df.groupby('Test_ID').size().unique().tolist())}"
)
print(
    "RH levels            : "
    f"{sorted(validation_df['RH_Setpoint'].unique().tolist())}"
)
print(
    "Temperature levels C : "
    f"{sorted(validation_df['Temperature_Setpoint_C'].unique().tolist())}"
)
print(
    "Directions           : "
    f"{sorted(validation_df['Direction'].astype(str).unique().tolist())}"
)
print(f"Days                 : {sorted(validation_df['Day_ID'].unique().tolist())}")
print(f"Runs                 : {sorted(validation_df['Run_ID'].unique().tolist())}")

assert len(validation_df) == 2160
assert validation_df["Test_ID"].nunique() == 36
assert validation_matrix_df["Test_ID"].nunique() == 36
assert sorted(validation_df.groupby("Test_ID").size().unique().tolist()) == [60]
assert sorted(validation_df["RH_Setpoint"].unique().tolist()) == [30.0, 50.0, 70.0]
assert sorted(validation_df["Temperature_Setpoint_C"].unique().tolist()) == [15.0, 25.0, 35.0]
assert {str(x).upper() for x in validation_df["Direction"].unique()} == {"UP", "DOWN"}
assert sorted(validation_df["Day_ID"].unique().tolist()) == [3]
assert sorted(validation_df["Run_ID"].unique().tolist()) == [1, 2]

print("\nIV-01 VALIDATION boundary audit: PASS")

VALIDATION boundary audit
-------------------------
Measurement rows      : 2160
Measurement Test_IDs  : 36
Matrix Test_IDs       : 36
Samples/Test_ID      : [60]
RH levels            : [30.0, 50.0, 70.0]
Temperature levels C : [15.0, 25.0, 35.0]
Directions           : ['DOWN', 'UP']
Days                 : [3]
Runs                 : [1, 2]

IV-01 VALIDATION boundary audit: PASS


## 5. IV-02 — Frozen Model Artifact Verification

The model is loaded from the version-controlled HV-006 JSON artifact.

The notebook does **not** recompute or refit the C1 coefficient.

In [3]:
c1_record, c5_record = load_frozen_calibration_records(project_root)

assert c1_record["model_id"] == "C1_Constant_Offset"
assert c1_record["validation_measurements_used"] is False
assert c5_record["model_id"] == "C5_Direction_Diagnostic"
assert c5_record["validation_measurements_used"] is False

frozen_c1_offset = float(
    c1_record["parameters"]["offset_percent_rh"]
)
frozen_c5_down = float(
    c5_record["parameters"]["intercept_down_correction_percent_rh"]
)
frozen_c5_up = float(
    c5_record["parameters"]["up_correction_percent_rh"]
)

assert np.isclose(frozen_c1_offset, 1.2266788194444438)
assert np.isclose(frozen_c5_down, 0.8164583333333324)
assert np.isclose(frozen_c5_up, 1.6368993055555552)

print(f"Frozen C1 offset      : {frozen_c1_offset:.10f} %RH")
print(f"Frozen C5 DOWN corr.  : {frozen_c5_down:.10f} %RH")
print(f"Frozen C5 UP corr.    : {frozen_c5_up:.10f} %RH")
print("\nIV-02 frozen model verification: PASS")

Frozen C1 offset      : 1.2266788194 %RH
Frozen C5 DOWN corr.  : 0.8164583333 %RH
Frozen C5 UP corr.    : 1.6368993056 %RH

IV-02 frozen model verification: PASS


## 6. IV-03 — Test_ID-Level Validation Summary

One Test_ID summary is the primary independent validation record.

The 60 within-Test_ID samples are repeated samples under the same condition and are not treated as independent validation experiments.

In [4]:
validation_summary_df = summarize_validation_test_ids(validation_df)

design_summary = {
    "N_Test_IDs": int(validation_summary_df["Test_ID"].nunique()),
    "RH_Levels": sorted(validation_summary_df["RH_Setpoint"].unique().tolist()),
    "Temperature_Levels_C": sorted(
        validation_summary_df["Temperature_Setpoint_C"].unique().tolist()
    ),
    "Directions": sorted(
        validation_summary_df["Direction"].astype(str).unique().tolist()
    ),
    "Days": sorted(validation_summary_df["Day_ID"].unique().tolist()),
    "Runs": sorted(validation_summary_df["Run_ID"].unique().tolist()),
    "Samples_Per_Test_ID": sorted(
        validation_summary_df["N_Samples"].unique().tolist()
    ),
}

for key, value in design_summary.items():
    print(f"{key:24s}: {value}")

assert design_summary["N_Test_IDs"] == 36
assert design_summary["RH_Levels"] == [30.0, 50.0, 70.0]
assert design_summary["Temperature_Levels_C"] == [15.0, 25.0, 35.0]
assert {x.upper() for x in design_summary["Directions"]} == {"UP", "DOWN"}
assert design_summary["Days"] == [3]
assert design_summary["Runs"] == [1, 2]
assert design_summary["Samples_Per_Test_ID"] == [60]

print("\nIV-03 Test_ID summary audit: PASS")

N_Test_IDs              : 36
RH_Levels               : [30.0, 50.0, 70.0]
Temperature_Levels_C    : [15.0, 25.0, 35.0]
Directions              : ['DOWN', 'UP']
Days                    : [3]
Runs                    : [1, 2]
Samples_Per_Test_ID     : [60]

IV-03 Test_ID summary audit: PASS


## 7. Apply Frozen C1 and Diagnostic C5

C0 is the uncalibrated validation baseline.

C1 is the primary frozen deployable model.

C5 is applied only as a diagnostic comparator to test whether the Direction/hysteresis effect independently reproduces.

In [5]:
validation_results_df = validation_summary_df.copy()

# C0 — uncalibrated
validation_results_df["Residual_C0"] = validation_results_df["Error_RH_Pre"]
validation_results_df["RH_Calibrated_C0"] = validation_results_df["RH_DUT_Mean"]

# C1 — frozen deployable constant offset
validation_results_df["Predicted_Correction_C1"] = frozen_c1_offset
validation_results_df["RH_Calibrated_C1"] = (
    validation_results_df["RH_DUT_Mean"]
    + validation_results_df["Predicted_Correction_C1"]
)
validation_results_df["Residual_C1"] = (
    validation_results_df["RH_Calibrated_C1"]
    - validation_results_df["RH_Reference_Mean"]
)

# C5 — frozen diagnostic Direction correction
direction_upper = validation_results_df["Direction"].astype(str).str.upper()
validation_results_df["Predicted_Correction_C5"] = np.where(
    direction_upper.eq("UP"),
    frozen_c5_up,
    frozen_c5_down,
)
validation_results_df["RH_Calibrated_C5"] = (
    validation_results_df["RH_DUT_Mean"]
    + validation_results_df["Predicted_Correction_C5"]
)
validation_results_df["Residual_C5"] = (
    validation_results_df["RH_Calibrated_C5"]
    - validation_results_df["RH_Reference_Mean"]
)

assert np.allclose(
    validation_results_df["Residual_C0"],
    validation_results_df["Error_RH_Pre"],
)

print("Frozen models applied without fitting.")

Frozen models applied without fitting.


## 8. IV-04 — Independent Validation Metrics

Primary metric: RMSE.

Also report MBE, MAE, median error, residual SD, maximum absolute error, and R².

No metric is used to refit the frozen model.

In [6]:
def validation_metrics(
    residual: pd.Series | np.ndarray,
    rh_reference: pd.Series | np.ndarray,
    rh_calibrated: pd.Series | np.ndarray,
) -> dict[str, float]:
    r = np.asarray(residual, dtype=float)
    ref = np.asarray(rh_reference, dtype=float)
    cal = np.asarray(rh_calibrated, dtype=float)

    return {
        "N": int(r.size),
        "MBE": float(np.mean(r)),
        "MAE": float(np.mean(np.abs(r))),
        "RMSE": float(np.sqrt(np.mean(r**2))),
        "Median_Error": float(np.median(r)),
        "Residual_SD": float(np.std(r, ddof=1)),
        "Max_Abs_Error": float(np.max(np.abs(r))),
        "R2": float(r2_score(ref, cal)),
    }

c0_val_metrics = validation_metrics(
    validation_results_df["Residual_C0"],
    validation_results_df["RH_Reference_Mean"],
    validation_results_df["RH_Calibrated_C0"],
)
c1_val_metrics = validation_metrics(
    validation_results_df["Residual_C1"],
    validation_results_df["RH_Reference_Mean"],
    validation_results_df["RH_Calibrated_C1"],
)
c5_val_metrics = validation_metrics(
    validation_results_df["Residual_C5"],
    validation_results_df["RH_Reference_Mean"],
    validation_results_df["RH_Calibrated_C5"],
)

validation_model_comparison_df = pd.DataFrame(
    [
        {"Model": "C0_Uncalibrated", "Role": "Baseline", **c0_val_metrics},
        {"Model": "C1_Frozen_Deployable", "Role": "Primary", **c1_val_metrics},
        {"Model": "C5_Frozen_Diagnostic", "Role": "Diagnostic", **c5_val_metrics},
    ]
)

validation_model_comparison_df["RMSE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_val_metrics["RMSE"]
        - validation_model_comparison_df["RMSE"]
    )
    / c0_val_metrics["RMSE"]
)
validation_model_comparison_df["MAE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_val_metrics["MAE"]
        - validation_model_comparison_df["MAE"]
    )
    / c0_val_metrics["MAE"]
)

display(validation_model_comparison_df.round(4))

,Model,Role,N,MBE,MAE,RMSE,Median_Error,Residual_SD,Max_Abs_Error,R2,RMSE_Improvement_vs_C0_pct,MAE_Improvement_vs_C0_pct
0,C0_Uncalibrated,Baseline,36,-1.0583,1.0583,1.1403,-1.0652,0.4305,1.6960,0.9951,0.0000,0.0000
1,C1_Frozen_Deployable,Primary,36,0.1683,0.4046,0.4566,0.1615,0.4305,0.8275,0.9992,59.9567,61.7664
2,C5_Frozen_Diagnostic,Diagnostic,36,0.1683,0.1792,0.2131,0.1711,0.1325,0.4236,0.9998,81.3153,83.0685


## 9. Compare C1 With Frozen HV-006 Development Evidence

This comparison is descriptive only.

The validation result cannot trigger model retuning.

In [7]:
development_reference_df = pd.DataFrame(
    [
        {
            "Assessment": "HV-006 Primary GroupCV",
            "C1_RMSE": float(
                c1_record["selection_summary"][
                    "primary_groupcv_rmse_percent_rh"
                ]
            ),
        },
        {
            "Assessment": "HV-006 LOO-RH",
            "C1_RMSE": float(
                c1_record["selection_summary"]["loo_rh_rmse_percent_rh"]
            ),
        },
        {
            "Assessment": "HV-006 LOO-Day",
            "C1_RMSE": float(
                c1_record["selection_summary"]["loo_day_rmse_percent_rh"]
            ),
        },
        {
            "Assessment": "HV-007 Blind Validation",
            "C1_RMSE": float(c1_val_metrics["RMSE"]),
        },
    ]
)

display(development_reference_df.round(4))

,Assessment,C1_RMSE
0,HV-006 Primary GroupCV,0.5216
1,HV-006 LOO-RH,0.4535
2,HV-006 LOO-Day,0.4505
3,HV-007 Blind Validation,0.4566


## 10. IV-05 — Residual Structure Diagnostics

Frozen C1 residuals are summarized by RH, temperature, Direction, and Run.

C5 is included in the same summaries only as a diagnostic comparator.

In [8]:
def grouped_residual_summary(group_column: str) -> pd.DataFrame:
    return (
        validation_results_df
        .groupby(group_column, as_index=False)
        .agg(
            N=("Test_ID", "size"),
            C0_MBE=("Residual_C0", "mean"),
            C1_MBE=("Residual_C1", "mean"),
            C5_MBE=("Residual_C5", "mean"),
            C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
            C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
            C5_MAE=("Residual_C5", lambda x: x.abs().mean()),
        )
    )

residual_by_rh_df = grouped_residual_summary("RH_Setpoint")
residual_by_temperature_df = grouped_residual_summary("Temperature_Setpoint_C")
residual_by_direction_df = grouped_residual_summary("Direction")
residual_by_run_df = grouped_residual_summary("Run_ID")

print("Residual structure by RH")
display(residual_by_rh_df.round(4))

print("Residual structure by temperature")
display(residual_by_temperature_df.round(4))

print("Residual structure by Direction")
display(residual_by_direction_df.round(4))

print("Residual structure by Run")
display(residual_by_run_df.round(4))

Residual structure by RH


,RH_Setpoint,N,C0_MBE,C1_MBE,C5_MBE,C0_MAE,C1_MAE,C5_MAE
0,30.0,12,-1.1341,0.0926,0.0926,1.1341,0.4024,0.0986
1,50.0,12,-0.9846,0.2421,0.2421,0.9846,0.4058,0.2421
2,70.0,12,-1.0564,0.1703,0.1703,1.0564,0.4057,0.1969


Residual structure by temperature


,Temperature_Setpoint_C,N,C0_MBE,C1_MBE,C5_MBE,C0_MAE,C1_MAE,C5_MAE
0,15.0,12,-0.9913,0.2353,0.2353,0.9913,0.4016,0.2413
1,25.0,12,-1.0639,0.1628,0.1628,1.0639,0.4156,0.1628
2,35.0,12,-1.1198,0.1069,0.1069,1.1198,0.3968,0.1335


Residual structure by Direction


,Direction,N,C0_MBE,C1_MBE,C5_MBE,C0_MAE,C1_MAE,C5_MAE
0,DOWN,18,-0.6544,0.5722,0.1620,0.6544,0.5722,0.1772
1,UP,18,-1.4623,-0.2356,0.1746,1.4623,0.2371,0.1812


Residual structure by Run


,Run_ID,N,C0_MBE,C1_MBE,C5_MBE,C0_MAE,C1_MAE,C5_MAE
0,1,18,-1.0015,0.2252,0.2252,1.0015,0.4132,0.2252
1,2,18,-1.1152,0.1115,0.1115,1.1152,0.3961,0.1332


## 11. Curated HV-007 Evidence Export

In [9]:
validation_design_summary_df = pd.DataFrame(
    [
        {"Quantity": key, "Value": str(value)}
        for key, value in design_summary.items()
    ]
)

validation_design_summary_df.to_csv(
    tables_dir / "hv007_validation_design_summary.csv",
    index=False,
)
validation_model_comparison_df.to_csv(
    tables_dir / "hv007_validation_model_comparison.csv",
    index=False,
)
residual_by_rh_df.to_csv(
    tables_dir / "hv007_validation_residual_by_rh.csv",
    index=False,
)
residual_by_temperature_df.to_csv(
    tables_dir / "hv007_validation_residual_by_temperature.csv",
    index=False,
)
residual_by_direction_df.to_csv(
    tables_dir / "hv007_validation_residual_by_direction.csv",
    index=False,
)
residual_by_run_df.to_csv(
    tables_dir / "hv007_validation_residual_by_run.csv",
    index=False,
)

print("Curated HV-007 tables exported.")

Curated HV-007 tables exported.


## 12. Initial IV-001 Checkpoint

A clean-kernel run should establish:

- `IV-01 VALIDATION boundary audit: PASS`;
- `IV-02 frozen model verification: PASS`;
- `IV-03 Test_ID summary audit: PASS`;
- 36 independent Test_ID-level validation records;
- C0, frozen C1, and frozen diagnostic C5 metrics;
- C1 comparison with frozen HV-006 evidence;
- residual structure by RH, temperature, Direction, and Run;
- curated HV-007 CSV exports.

### Do not modify the C1 coefficient after viewing these results.

The next decision is interpretation of independent generalization evidence and formal IV-001 gate closure.

## Validation-design scope

VALIDATION measurements are blind to model development, but they are not a fully crossed external campaign. CALIBRATION uses Days 1–2 at RH 20/40/60/80 %RH; VALIDATION uses only Day 3 at RH 30/50/70 %RH.

Therefore this notebook demonstrates generalization to a new day and interpolated RH points inside the tested range. Day 3 and validation role are confounded, so the observed validation residual cannot be separated into a pure day effect and a pure model-generalization effect.

## 13. Engineering Interpretation of Independent Validation

The blind-validation results support the frozen C1 calibration model without any refitting or retuning.

### Independent improvement versus uncalibrated C0

On 36 reserved VALIDATION Test_IDs:

- C0 RMSE = **1.1403 %RH**;
- frozen C1 RMSE = **0.4566 %RH**;
- C1 RMSE improvement = **59.96%**;
- C0 MAE = **1.0583 %RH**;
- frozen C1 MAE = **0.4046 %RH**;
- C1 MAE improvement = **61.77%**.

The frozen model therefore reproduces almost the same approximately 60% error reduction observed during HV-006 development.

### Agreement with HV-006 development evidence

C1 RMSE:

```text
HV-006 Primary GroupCV  0.5216 %RH
HV-006 LOO-RH           0.4535 %RH
HV-006 LOO-Day          0.4505 %RH
HV-007 Blind Validation 0.4566 %RH
```

The blind-validation RMSE is approximately:

- 12% lower than the primary GroupCV estimate;
- less than 1% different from the LOO-RH estimate;
- about 1% different from the LOO-Day estimate.

This is strong evidence that C1 generalizes consistently to the reserved static validation set.

### Remaining global center shift

Frozen C1 validation MBE is **+0.1683 %RH**.

The original validation MBE is **-1.0583 %RH**. The frozen +1.2266788194 %RH correction therefore removes most of the negative bias but slightly over-corrects the validation center.

The C1 coefficient remains frozen and is not adjusted to remove this validation offset.

### Independent reproduction of Direction / hysteresis

After C1:

```text
DOWN MBE = +0.5722 %RH
UP MBE   = -0.2356 %RH
```

The DOWN-UP residual separation is approximately **0.8078 %RH**.

This is very close to the frozen C5 Direction increment of approximately **0.8204 %RH** from HV-006.

Therefore the dominant Direction/hysteresis structure identified during calibration development independently reproduces in the blind validation data.

### Frozen C5 diagnostic

C5 remains diagnostic only, but independently gives:

- RMSE = **0.2131 %RH**;
- MAE = **0.1792 %RH**;
- DOWN MBE = **+0.1620 %RH**;
- UP MBE = **+0.1746 %RH**.

Thus C5 nearly removes the Direction split, while the remaining positive center shift persists.

This supports the interpretation that two structures coexist:

1. a large Direction/state-dependent hysteresis term;
2. a smaller validation-wide positive residual center shift.

C5 does not replace C1 as the deployable model because Direction was not authorized as a reliable field input before validation exposure.

## 14. IV-001 Gate Decision

**IV-001 = PASS WITH DOCUMENTED CONDITIONS**

### Basis for PASS

- the VALIDATION boundary is verified;
- the frozen C1 artifact was loaded unchanged;
- no refitting or retuning occurred;
- C1 reduces blind-validation RMSE by approximately 60%;
- C1 blind-validation RMSE is highly consistent with the pre-validation HV-006 robustness estimates;
- the known Direction/hysteresis limitation independently reproduces;
- frozen C5 independently confirms the value of Direction/state information;
- no validation result is used to change the C1 identity or coefficient.

### Documented conditions

1. **Residual global bias:** frozen C1 leaves a validation MBE of approximately +0.1683 %RH.
2. **Direction/hysteresis:** memoryless C1 retains a substantial Direction-dependent residual split of approximately 0.8078 %RH.
3. **Deployment scope:** C5 remains diagnostic/state-aware and may not replace C1 unless Direction can later be operationally and reliably determined.
4. **Scope of conclusion:** HV-007 validates static calibration generalization only. Final sensor suitability remains open pending dynamic response, stability/drift, and measurement uncertainty.

The next engineering stage is:

```text
HV-008 — Dynamic Step Response
```